In [14]:
import json
import os
import random

OUTPUT_FOLDER = r"D:\PJ\spatial_dataset"


def get_answers(obj):

    return {
        "speaker": obj["relations"]["speaker"],
        "intrinsic": obj["relations"]["intrinsic"],
        "hearer": obj["relations"]["hearer"]
    }


def check_conflict_pairs(data):

    objects = data["objects"]
    landmark_rotation = data["landmark"]["rotation"]

    if landmark_rotation == 0:

        if objects["front"]["type"] != objects["right"]["type"]:
            raise ValueError(
                f"{data['scene_id']}: rotation=0 时，front 和 right 应该是同一种物体。"
            )

        if objects["back"]["type"] != objects["left"]["type"]:
            raise ValueError(
                f"{data['scene_id']}: rotation=0 时，back 和 left 应该是同一种物体。"
            )

    elif landmark_rotation == 180:

        if objects["front"]["type"] != objects["left"]["type"]:
            raise ValueError(
                f"{data['scene_id']}: rotation=180 时，front 和 left 应该是同一种物体。"
            )

        if objects["back"]["type"] != objects["right"]["type"]:
            raise ValueError(
                f"{data['scene_id']}: rotation=180 时，back 和 right 应该是同一种物体。"
            )

    else:

        raise ValueError(
            f"{data['scene_id']}: Landmark rotation 必须是 0 或 180。"
        )


def generate_experiment1(data):

    objects = data["objects"]

    questions = []

    for role in ["front", "back", "left", "right"]:

        obj = objects[role]

        questions.append({

            "question": (
                f"Where is the {obj['color']} {obj['type']}?"
            ),

            "target": obj["type"],

            "color": obj["color"],

            "role": role,

            "answers": get_answers(obj)

        })

    return {

        "experiment": 1,

        "questions": questions

    }


def generate_experiment2(data):

    landmark_type = data["landmark"]["type"]

    objects = data["objects"]

    roles = [
        "front",
        "back",
        "left",
        "right"
    ]

    target_role = random.choice(roles)

    obj = objects[target_role]

    object_type = obj["type"]

    color = obj["color"]

    answers = get_answers(obj)

    questions = [

        {
            "question": (
                f"From the speaker's perspective, "
                f"where is the {color} {object_type} "
                f"relative to the {landmark_type}?"
            ),

            "target": object_type,

            "color": color,

            "FoR": "speaker",

            "answers": answers

        },

        {
            "question": (
                f"From the {landmark_type}'s own orientation, "
                f"where is the {color} {object_type} "
                f"relative to the {landmark_type}?"
            ),

            "target": object_type,

            "color": color,

            "FoR": "intrinsic",

            "answers": answers

        },

        {
            "question": (
                f"From the hearer's perspective, "
                f"where is the {color} {object_type} "
                f"relative to the {landmark_type}?"
            ),

            "target": object_type,

            "color": color,

            "FoR": "hearer",

            "answers": answers

        }

    ]

    return {

        "experiment": 2,

        "landmark": {
            "type": landmark_type
        },

        "target": {
            "role": target_role,
            "type": object_type,
            "color": color
        },

        "questions": questions

    }


def generate_experiment3(data):

    check_conflict_pairs(data)

    landmark_type = data["landmark"]["type"]

    objects = data["objects"]

    priming = objects["priming"]

    evaluation = objects["evaluation"]

    right_item = objects["right"]

    left_item = objects["left"]

    t1 = (
        f"Can you see the {priming['type']} in the scene?"
    )

    t2 = (
        f"What is the color of the "
        f"{right_item['type']} to the right of the {landmark_type}?"
    )

    t3 = (
        f"What is the color of the "
        f"{left_item['type']} to the left of the {landmark_type}?"
    )

    t4 = (
        f"Where is the {evaluation['type']} "
        f"relative to the {landmark_type}?"
    )

    return {

        "experiment": 3,

        "T1": {

            "question": t1,

            "object": priming["type"],

            "color": priming["color"],

            "role": "priming",

            "answers": get_answers(priming)

        },

        "T2": {

            "question": t2,

            "object": right_item["type"],

            "color": right_item["color"],

            "role": "right",

            "direction": "right",

            "answers": get_answers(right_item)

        },

        "T3": {

            "question": t3,

            "object": left_item["type"],

            "color": left_item["color"],

            "role": "left",

            "direction": "left",

            "answers": get_answers(left_item)

        },

        "T4": {

            "question": t4,

            "object": evaluation["type"],

            "color": evaluation["color"],

            "role": "evaluation",

            "answers": get_answers(evaluation)

        }

    }


def generate_questions_dataset():

    scene_files = []

    for filename in os.listdir(OUTPUT_FOLDER):

        if (
            filename.startswith("scene_")
            and filename.endswith(".json")
            and "_experiment" not in filename
        ):

            scene_files.append(filename)

    scene_files.sort()

    all_scenes = []

    for filename in scene_files:

        json_path = os.path.join(
            OUTPUT_FOLDER,
            filename
        )

        with open(
            json_path,
            "r",
            encoding="utf-8"
        ) as f:

            data = json.load(f)

        print(
            "Processing:",
            data["scene_id"]
        )

        experiment1 = generate_experiment1(data)

        experiment2 = generate_experiment2(data)

        experiment3 = generate_experiment3(data)

        scene_questions = {

            "scene_id": data["scene_id"],

            "image": data["scene_id"] + "_speaker.png",

            "experiment1": experiment1,

            "experiment2": experiment2,

            "experiment3": experiment3

        }

        all_scenes.append(
            scene_questions
        )

    dataset = {

        "dataset": "spatial_dataset",

        "num_scenes": len(all_scenes),

        "scenes": all_scenes

    }

    output_path = os.path.join(
        OUTPUT_FOLDER,
        "questions.json"
    )

    with open(
        output_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            dataset,
            f,
            indent=4,
            ensure_ascii=False
        )

    print()
    print("========================================")
    print("Question dataset generated")
    print("========================================")
    print("Number of scenes:", len(all_scenes))
    print("Output:", output_path)


generate_questions_dataset()

Processing: scene_0001

Question dataset generated
Number of scenes: 1
Output: D:\PJ\spatial_dataset\questions.json
